In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
import os
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_Res
from train import WarmUpCosine, CustomWeightDecaySGD
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_cla import load_wb_if_exists
from evaluate_cla import evalu_stream_main_selected, per_group_ovr_accuracy, evalu_select

In [4]:
with open('data.pkl', 'rb') as f:
    [x_train,y_train,x_test,y_test]=pickle.load(f)
y_train_onehot=tf.keras.utils.to_categorical(y_train,num_classes=4)
y_test_onehot=tf.keras.utils.to_categorical(y_test,num_classes=4)

In [5]:
model=load_Res()

In [6]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("re_lu_16").output,
    outputs=model.output
)

In [7]:
l_label=[3, 11, 18, 27, 34, 43, 50, 59, 66]

In [8]:
layer_list = [model.layers[l].name for l in l_label]

In [9]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data_3/Res-18/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data_3/Res-18/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data_3/Res-18/layer_cache/salt/"+str(i))

[SAVE] Generating layer outputs for: D:/Data_3/Res-18/layer_cache/base
[Saved] re_lu: outputs (20000, 32768), labels (20000, 1)
[Saved] re_lu_2: outputs (20000, 8192), labels (20000, 1)
[Saved] re_lu_4: outputs (20000, 8192), labels (20000, 1)
[Saved] re_lu_6: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_8: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_10: outputs (20000, 2048), labels (20000, 1)
[Saved] re_lu_12: outputs (20000, 2048), labels (20000, 1)
[Saved] re_lu_14: outputs (20000, 1024), labels (20000, 1)
[Saved] re_lu_16: outputs (20000, 1024), labels (20000, 1)
[SAVE] Generating layer outputs for: D:/Data_3/Res-18/layer_cache/gauss/0
[Saved] re_lu: outputs (20000, 32768), labels (20000, 1)
[Saved] re_lu_2: outputs (20000, 8192), labels (20000, 1)
[Saved] re_lu_4: outputs (20000, 8192), labels (20000, 1)
[Saved] re_lu_6: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_8: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_10: outputs (20000, 2048), 

In [10]:
CACHE_DIR = "./Res-18/w_and_b_cache"

In [11]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data_3/Res-18/layer_cache/base")


==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 14110
xi>=0 count: 14232
xi>=0 count: 12903
xi>=0 count: 14088

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 14041
xi>=0 count: 14434
xi>=0 count: 12886
xi>=0 count: 14224

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 14886
xi>=0 count: 14757
xi>=0 count: 12971
xi>=0 count: 14546

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 16442
xi>=0 count: 16428
xi>=0 count: 15036
xi>=0 count: 15331

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 16712
xi>=0 count: 17186
xi>=0 count: 15730
xi>=0 count: 16281

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 18709
xi>=0 count: 19021
xi>=0 count: 18722
xi>=0 count: 18693

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count:

In [12]:
NOISE_dir='./noise/'
ATTACK_dir=Noise_dir='./attack/'

In [13]:
pred_model.input_shape

(None, 2, 2, 256)

In [14]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir="D:/Data_3/Res-18/layer_cache/base")

[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]


In [15]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir="D:/Data_3/Res-18/layer_cache/base")
base_final = per_group_ovr_accuracy(model, x_train, y_train, select_list)
base = np.concatenate((base,base_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.66927626 0.53138692 0.61495033 0.56324461]
Layer 1
accuracy: [0.63552001 0.54965702 0.73851678 0.56284115]
Layer 2
accuracy: [0.67024215 0.54034754 0.62519551 0.57044746]
Layer 3
accuracy: [0.76973741 0.65308082 0.75696933 0.62706545]
Layer 4
accuracy: [0.79549802 0.68549352 0.76851389 0.63321798]
Layer 5
accuracy: [0.88999827 0.85478907 0.86594342 0.79746177]
Layer 6
accuracy: [0.93149348 0.96151861 0.89248512 0.97069037]
Layer 7
accuracy: [0.97050006 0.98227495 0.97224372 0.97870747]
Layer 8
accuracy: [0.9849963  0.96029527 0.97725413 0.9862219 ]


In [16]:
base

array([[0.66927626, 0.53138692, 0.61495033, 0.56324461],
       [0.63552001, 0.54965702, 0.73851678, 0.56284115],
       [0.67024215, 0.54034754, 0.62519551, 0.57044746],
       [0.76973741, 0.65308082, 0.75696933, 0.62706545],
       [0.79549802, 0.68549352, 0.76851389, 0.63321798],
       [0.88999827, 0.85478907, 0.86594342, 0.79746177],
       [0.93149348, 0.96151861, 0.89248512, 0.97069037],
       [0.97050006, 0.98227495, 0.97224372, 0.97870747],
       [0.9849963 , 0.96029527, 0.97725413, 0.9862219 ],
       [1.        , 1.        , 1.        , 1.        ]])

In [17]:
gauss = np.zeros((len(layer_list),4))
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_3/Res-18/layer_cache/gauss/"+str(i))
gauss = gauss/10
gauss_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/gauss.npy'
    x_noise = np.load(DIR)
    gauss_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.33676398 0.25121557 0.30119965 0.25008135]
Layer 1
accuracy: [0.46599937 0.32004821 0.74975933 0.33882801]
Layer 2
accuracy: [0.70775226 0.56705964 0.67797641 0.58573421]
Layer 3
accuracy: [0.75725865 0.71123349 0.75276944 0.67629269]
Layer 4
accuracy: [0.77276015 0.73076807 0.75501134 0.68398276]
Layer 5
accuracy: [0.81850767 0.80407019 0.79250125 0.76411695]
Layer 6
accuracy: [0.85926329 0.88658518 0.82027533 0.90232038]
Layer 7
accuracy: [0.88701199 0.89757759 0.8737502  0.88324527]
Layer 8
accuracy: [0.91275563 0.85630304 0.89371877 0.89867154]
Layer 0
accuracy: [0.3370017  0.25121985 0.30196703 0.24983602]
Layer 1
accuracy: [0.46399943 0.32830526 0.75001059 0.33659086]
Layer 2
accuracy: [0.70474223 0.56334202 0.67698117 0.58270303]
Layer 3
accuracy: [0.7587467  0.71077351 0.75126733 0.67606249]
Layer 4
accuracy: [0.774994   0.7260332  0.75401279 0.68778594]
Layer 5
accuracy: [0.82248253 0.79601947 0.7935235  0.7647391 ]
Layer 6
accuracy: [0.85126723 0.88510684

In [18]:
gauss

array([[0.33763762, 0.25069812, 0.30095217, 0.25006129],
       [0.46465601, 0.32133793, 0.74996079, 0.33454395],
       [0.70702256, 0.56666349, 0.676744  , 0.58387841],
       [0.75833105, 0.71154024, 0.75281722, 0.67704071],
       [0.77530498, 0.72999756, 0.75481391, 0.68625455],
       [0.82259995, 0.79915813, 0.79148726, 0.76747123],
       [0.85326539, 0.88935192, 0.82080171, 0.90164764],
       [0.88174578, 0.89849566, 0.875138  , 0.88239557],
       [0.90574942, 0.85950104, 0.89564906, 0.89813761],
       [0.976275  , 0.96997501, 0.95947499, 0.9607    ]])

In [19]:
salt = np.zeros((len(layer_list),4))
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_3/Res-18/layer_cache/salt/"+str(i))
salt = salt/10
salt_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/salt.npy'
    x_noise = np.load(DIR)
    salt_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.49127937 0.27306077 0.35347039 0.27389363]
Layer 1
accuracy: [0.61324396 0.48230906 0.75025468 0.49576619]
Layer 2
accuracy: [0.67225011 0.52652637 0.6189596  0.5569555 ]
Layer 3
accuracy: [0.76524454 0.67255587 0.75551848 0.64477155]
Layer 4
accuracy: [0.78450608 0.70054426 0.7635414  0.65474001]
Layer 5
accuracy: [0.85751102 0.83354285 0.8364988  0.78324596]
Layer 6
accuracy: [0.90125643 0.93575562 0.86028054 0.94924291]
Layer 7
accuracy: [0.93600454 0.95286303 0.93350847 0.94229931]
Layer 8
accuracy: [0.96076388 0.91799634 0.94697427 0.95929319]
Layer 0
accuracy: [0.49029502 0.2742604  0.35948543 0.28008372]
Layer 1
accuracy: [0.60974457 0.48036163 0.75152723 0.50450233]
Layer 2
accuracy: [0.66949185 0.52865519 0.61717628 0.56220799]
Layer 3
accuracy: [0.76649107 0.67282127 0.75528457 0.6438418 ]
Layer 4
accuracy: [0.78550335 0.70409448 0.76328073 0.6532562 ]
Layer 5
accuracy: [0.85926156 0.83256972 0.83850257 0.79526666]
Layer 6
accuracy: [0.89299883 0.93578324

In [20]:
salt

array([[0.49093231, 0.27455041, 0.35511584, 0.2748914 ],
       [0.60907502, 0.48044186, 0.75216049, 0.50292502],
       [0.67067203, 0.52633287, 0.61972936, 0.55988918],
       [0.7648475 , 0.67317989, 0.75498923, 0.64561302],
       [0.78665505, 0.7011186 , 0.76318404, 0.65368421],
       [0.86018158, 0.8330582 , 0.83633906, 0.78880066],
       [0.89578025, 0.9358266 , 0.85957849, 0.94574296],
       [0.93333502, 0.94973838, 0.92807817, 0.94127999],
       [0.95673378, 0.91515614, 0.94141687, 0.95863489],
       [0.99409999, 0.99412501, 0.98865   , 0.99205   ]])

In [21]:
SAVE_FILE='e-Res-18.pkl'

In [22]:
progress = {"base": base, "gauss": gauss,"salt": salt}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [23]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=3): 
    """ 
    Input: 
        matrix_dict: {name: np.ndarray(m, n)}, a dictionary containing 3 matrices 
    Output: 
        stats: { 
            name: { 
                "mean": (m,), 
                "min":  (m,), 
                "max":  (m,) 
            }, ... 
        } 
    For each component i, statistics are computed along axis=1 (over n samples): 
        mean[i] = mean(X[i, :]) 
        min[i]  = min(X[i, :]) 
        max[i]  = max(X[i, :]) 
    """ 
    if check_num is not None and len(matrix_dict) != check_num: 
        raise ValueError(f"Expected {check_num} matrices, but received {len(matrix_dict)}") 
 
    # shape consistency 
    shapes = [np.asarray(v).shape for v in matrix_dict.values()] 
    if len(set(shapes)) != 1: 
        raise ValueError(f"Inconsistent matrix shapes: {shapes}") 
 
    stats = {} 
    for name, X in matrix_dict.items(): 
        X = np.asarray(X) 
        stats[name] = { 
            "mean": X.mean(axis=1), 
            "std":  X.std(axis=1), 
            "min":  X.min(axis=1), 
            "max":  X.max(axis=1), 
        } 
    return stats 

In [25]:
mean_var = stats_minmax_from_matrix_dict(progress)

ValueError: 期望 6 个矩阵，但收到 3 个

In [ ]:
mean_var